In [22]:

def load_document(filepath):
    with open(filepath, "r") as f:
        content=f.read()
        return content
        
def chunk_by_section (text):
    import re 
    pattern=r"(Section \d+:.*?)(?=Section \d+:|\Z)"
    matches=re.findall(pattern,text,flags=re.DOTALL)
    
    chunks=[]

    for i, match in enumerate(matches):

        lines=match.split("\n",1)
        chunked_sections={
            "id":f"chunk{i}",
            "title":lines[0],
            "text":match
        }
        chunks.append(chunked_sections)
    return chunks

def build_vector_store(chunks,collection_name="policy_docs"):
    import chromadb
    client=chromadb.PersistentClient(path="chromadb")
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection=client.create_collection(collection_name)

    ids=[c["id"] for c in chunks]
    documents=[c["text"] for c in chunks]
    metadata=[ {"title":c["title"]} for c in chunks]

    collection.add(ids=ids,documents=documents,metadatas=metadata)
    return collection

def retrieve(query,collection_name="policy_docs",n_results=2):
    import chromadb
    client=chromadb.PersistentClient(path="chromadb")
    
    collection=client.get_collection(collection_name)
    results=collection.query(query_texts=[query],n_results=n_results)
    return results


   

def compare_policy_to_regulation(policy_text, regulation_text):
    import os
    from groq import Groq
    from dotenv import load_dotenv

    load_dotenv()
    
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    
    prompt = f"""
    You are a compliance analyst. Compare the COMPANY POLICY below against 
    the REGULATION below. Identify any gaps or conflicts, and explain why 
    they matter. Be specific and concise.
    
    COMPANY POLICY:
    {policy_text}
    
    REGULATION:
    {regulation_text}
    """
    response=client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )
    return response.choices[0].message.content
regulation_text="GDPR requires explicit opt-in consent before collecting personal data. Passive/implied consent (e.g. opt-out only) is not sufficient."

tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_policy",
            "description": "Search the company's policy documents for sections relevant to a topic or question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The topic or question to search for in the policy documents"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_regulations",
            "description": "Search the web for current laws, regulations, or compliance requirements on a given topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The regulation or legal topic to search for, e.g. 'GDPR data consent requirements'"
                    }
                },
                "required": ["topic"]
            }
        }
    }
]

def search_regulations(topic):
    import os
    from tavily import TavilyClient
    from dotenv import load_dotenv

    load_dotenv()
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    response = client.search(query=topic, max_results=3)

    summary=""
    for r in response ['results']:
        summary +=  f"Title: {r['title']}\nContent: {r['content']}\n url:{r['url']}"
    return summary
    
    
def ask_agent(user_message):
    import os
    from groq import Groq
    from dotenv import load_dotenv
    
    load_dotenv()
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": user_message}],
            tools=tools
        )
    except Exception as e:
        print("Agent failed to respond:", e)
        return
    
    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        import json
        args= json.loads(tool_call.function.arguments)
        print("Parsed arguments:", args)
        if tool_call.function.name=="retrieve_policy":
            results=retrieve(args["query"])
            print(results["documents"])
        elif tool_call.function.name=="search_regulations":
            results=search_regulations(args["topic"])
            print(results)
    else:
        print("Model answered directly:", message.content)
    
if __name__ == "__main__":
    text = load_document("data/sample_policy.txt")
    print("loaded document")
    chunks = chunk_by_section(text)
    print("chunks", len(chunks))
    collection = build_vector_store(chunks)
    print("stored", collection.count(), "chunks in vector db")

    ask_agent("Do we get explicit consent before collecting user data?")
    ask_agent("What are the current GDPR fines for non-compliance in 2026?")

    
    



loaded document
chunks 5
stored 5 chunks in vector db
Parsed arguments: {'query': 'explicit consent for user data collection'}
[['Section 1: Data Collection\nOur company collects user data including name, email, and browsing behavior for the purpose of improving our services. Users are notified of data collection through a banner on first visit. We do not currently require explicit opt-in consent before collecting behavioral data; users are considered opted-in unless they navigate to settings and opt out manually.\n\n', 'Section 3: Data Sharing with Third Parties\nWe may share aggregated, anonymized user data with marketing partners. We do not currently disclose the specific list of third parties receiving this data in our public privacy policy.\n\n']]
Parsed arguments: {'topic': 'GDPR fines for non-compliance 2026'}
Title: DLA Piper GDPR Fines and Data Breach Survey: January 2026 | DLA Piper
Content: The eighth annual edition of DLA Piper’s GDPR Fines and Data Breach Survey has reveal